<a href="https://colab.research.google.com/github/sweet-cross/cross_notebooks/blob/main/notebooks/workshop_20260309_assumptions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
if 'google.colab' in sys.modules:
    # Installs the repo as a package, along with its dependencies
    !pip install git+https://github.com/sweet-cross/cross_notebooks.git

# CROSS Plots

This notebook allows you to replicate the plots presenting the CROSS results.

## Packages and options

In [18]:
from crosscontract import CrossClient
import getpass
import pandas as pd
from typing import Self

## Connecting to CROSS

In [11]:
username = input("Enter your username or mail address: ")
password = getpass.getpass("Enter your password: ")
my_client = CrossClient(username=username, password=password)

In [15]:
cr = my_client.contracts.get("result_storage_output")
cr.contract.title

'Result submission - Storage output'

In [25]:
from crosscontract.crossclient.services import ContractResource
class ResultVariable:

    def __init__(
        self,
        contract_resource: ContractResource,
        label: str | None = None,
        unit: str | None = None,
    ):
        self._contract_resource = contract_resource
        self.contract_name = contract_resource.contract.name
        self.label = label or self._clean_title(contract_resource.contract.title)
        self.unit = unit

    def __repr__(self):
        return (f"ResultVariable(contract_name='{self.contract_name}', " 
                f"label='{self.label}', unit='{self.unit}')")
    @property
    def contract_resource(self):
        """ContractResource as obtained from CROSS platform"""
        return self._contract_resource

    @classmethod
    def from_client(
        cls,
        client: CrossClient,
        contract_name: str,
        label: str | None = None,
        **kwargs,
    ) -> Self:
        """Build from a contract fetched via the client.

        Label is derived from the contract title unless explicitly overridden.
        
        Args:
            client: An instance of CrossClient to fetch contract details.
            contract_name: The name of the contract to fetch.
        """
        cr = client.contracts.get(contract_name)
        return cls(contract_resource=cr, label=label, **kwargs)

    @property
    def description(self) -> str:
        """A human-readable description of the variable"""
        return self.contract_resource.contract.description

    @property
    def title(self) -> str:
        """A human-readable title of the variable"""
        return self.contract_resource.contract.title

    @staticmethod
    def _clean_title(title: str) -> str:
        """Strip common prefixes like 'Result submission - ' from contract titles.
        
        Args:
            title: The original contract title.
        Returns:
            The cleaned title with known prefixes removed."""
        prefixes = ["Result submission - ", "Result Submission - "]
        for prefix in prefixes:
            if title.startswith(prefix):
                return title[len(prefix):]
        return title

    def get_data(
        self,
        scenario_group: str = "cross202506",
        scenario_name: str | None = None,
        scenario_variant: str | None = None,
        model: str | None = None,
    ) -> pd.DataFrame:
        """Get CROSS 2025 data for a specified contract.

        Args:
            scenario_group (str, optional): The name of the scenario group to filter results by.
                Defaults to "cross202506".
            scenario_name (str | None, optional): The name of the scenario to filter results by.
                Defaults to None, which means no filtering by scenario name.
            scenario_variant (str | None, optional): The name of the scenario variant to filter results by.
                Defaults to None, which means no filtering by scenario variant.
            model (str | None, optional): The name of the model to filter results by.
                Defaults to None, which means no filtering by model.
                Only valid in case of a result contract.

        Returns:
            pd.DataFrame: A DataFrame containing the results for the specified contract.
        """
        filters = {"scenario_group": scenario_group}
        if scenario_name is not None:
            filters["scenario_name"] = scenario_name
        if scenario_variant is not None:
            filters["scenario_variant"] = scenario_variant
        if self.contract_name.startswith("result_") and model is not None:
            filters["model"] = model
        df = (
            self.contract_resource.get_data(filters=filters)
            .drop(columns=["scenario_group"])
        )
        return df


def build_registry(client) -> dict[str, ResultVariable]:
    contracts = [
        ("result_electricity_consumption", {}),
        ("result_electricity_supply", {}),
        ("result_h2_fec", {"label": "H₂ Final Energy Consumption"}),
        ("result_h2_supply", {"label": "H₂ Supply"}),
        ("result_liquids_consumption", {}),
        ("result_liquids_supply", {}),
        ("result_methane_consumption", {}),
        ("result_methane_supply", {}),
        ("result_process_heat_energy_production", {}),
        ("result_space_heat_energy_supply", {}),
        ("result_district_heat_energy_production", {}),
        ("result_passenger_road_public_fec", {}),
        ("result_passenger_road_private_fec", {}),
        ("result_freight_road_fec", {}),
        ("result_storage_installed_volume", {}),
        ("result_storage_output", {}),
        ("result_elec_cons_typical_day", {}),
        ("result_elec_supply_typical_day", {}),
    ]
    return {
        "_".join(name.split("_")[1:]): ResultVariable.from_client(client, name, **overrides)
        for name, overrides in contracts
    }

variable_registry = build_registry(my_client)

In [26]:
variable_registry["electricity_consumption"]

ResultVariable(contract_name='result_electricity_consumption', label='Electricity consumption', unit='None')

In [28]:
df_ = variable_registry["electricity_consumption"].get_data()
df_

,model,scenario_name,scenario_variant,sector,country,year,unit,value
0,ses,abroad-res-full,reference,grid_losses,CH,2050,TWh,0.000000e+00
1,ses,abroad-res-full,reference,battery_in,CH,2050,TWh,2.133964e-03
2,ses,abroad-res-full,reference,phs_in,CH,2050,TWh,1.574937e-02
3,ses,abroad-res-full,reference,passenger,CH,2050,TWh,1.445502e+01
4,ses,abroad-res-full,reference,road_public,CH,2050,TWh,1.281656e-04
...,...,...,...,...,...,...,...,...
1519,stem,domestic-nores-lim,reference,exports,CH,2050,TWh,1.291372e+01
1520,stem,domestic-nores-lim,reference,dac,CH,2030,TWh,0.000000e+00
1521,stem,domestic-nores-lim,reference,dac,CH,2035,TWh,0.000000e+00
1522,stem,domestic-nores-lim,reference,dac,CH,2040,TWh,8.890966e-07


In [29]:
def _plot_plotly(
    df,
    model_col,
    scenario_col,
    stack_col,
    value_col,
    x_label,
    colors,
    bar_height=0.35,
    group_gap=0.6,
    height=None,
    **kwargs,
):
    import plotly.graph_objects as go

    models = df[model_col].unique()
    scenarios = df[scenario_col].unique()
    techs = df[stack_col].unique()

    n_scenarios = len(scenarios)
    n_models = len(models)
    group_height = n_scenarios * bar_height + group_gap

    if height is None:
        height = max(400, int(n_models * group_height * 100))

    # Pre-compute y positions
    y_positions = {}  # (model, scenario) -> y
    for i, model in enumerate(models):
        group_base = i * group_height
        for j, scenario in enumerate(scenarios):
            y_positions[(model, scenario)] = group_base + j * bar_height

    # One trace per technology (for a unified legend)
    fig = go.Figure()

    for tech in techs:
        y_vals = []
        x_vals = []
        customdata = []

        for model in models:
            for scenario in scenarios:
                subset = df[
                    (df[model_col] == model)
                    & (df[scenario_col] == scenario)
                    & (df[stack_col] == tech)
                ]
                val = subset[value_col].sum() if len(subset) > 0 else 0
                y_vals.append(y_positions[(model, scenario)])
                x_vals.append(val)
                customdata.append((model, scenario))

        fig.add_trace(
            go.Bar(
                y=y_vals,
                x=x_vals,
                orientation="h",
                name=tech,
                marker_color=colors.get(tech, "#cccccc"),
                width=bar_height * 0.9,
                customdata=customdata,
                hovertemplate="%{customdata[0]} · %{customdata[1]}<br>"
                + tech
                + ": %{x}<extra></extra>",
            )
        )

    # Stacked mode
    fig.update_layout(barmode="stack")

    # Y-axis: scenario labels on the left
    all_y = []
    all_labels = []
    for model in models:
        for scenario in scenarios:
            all_y.append(y_positions[(model, scenario)])
            all_labels.append(scenario)

    fig.update_yaxes(
        tickvals=all_y,
        ticktext=all_labels,
        autorange="reversed",
    )

    # Right-side model annotations
    for i, model in enumerate(models):
        group_base = i * group_height
        y_center = group_base + (n_scenarios - 1) * bar_height / 2
        fig.add_annotation(
            x=1.02,
            y=y_center,
            xref="paper",
            yref="y",
            text=f"<b>{model}</b>",
            showarrow=False,
            xanchor="left",
            font=dict(size=12),
        )

    fig.update_layout(
        height=height,
        xaxis_title=x_label,
        legend=dict(
            orientation="h",
            y=-0.12,
            x=0.5,
            xanchor="center",
            font=dict(size=10),
        ),
        margin=dict(r=120),  # room for model labels
        template="plotly_white",
    )

    return fig

In [30]:
def _plot_mpl(
    df,
    model_col,
    scenario_col,
    stack_col,
    value_col,
    x_label,
    colors,
    bar_height=0.35,
    group_gap=0.6,
    figsize=None,
    **kwargs,
):
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    models = df[model_col].unique()
    scenarios = df[scenario_col].unique()
    techs = df[stack_col].unique()

    n_scenarios = len(scenarios)
    n_models = len(models)

    # y positions: models are groups, scenarios are bars within each group
    # model 0 occupies y positions [0, 1, ...n_scenarios-1], then gap, etc.
    group_height = n_scenarios * bar_height + group_gap

    if figsize is None:
        figsize = (10, max(4, n_models * group_height * 0.9))

    fig, ax = plt.subplots(figsize=figsize)

    y_ticks_model = []  # center of each model group  (right labels)
    y_ticks_scenario = []  # each bar position            (left labels)
    y_tick_scenario_labels = []

    for i, model in enumerate(models):
        group_base = i * group_height
        y_center_group = group_base + (n_scenarios - 1) * bar_height / 2

        y_ticks_model.append(y_center_group)

        for j, scenario in enumerate(scenarios):
            y = group_base + j * bar_height
            y_ticks_scenario.append(y)
            y_tick_scenario_labels.append(scenario)

            subset = df[(df[model_col] == model) & (df[scenario_col] == scenario)]
            left = 0.0
            for _, row in subset.iterrows():
                tech = row[stack_col]
                val = row[value_col]
                if val == 0:
                    continue
                ax.barh(
                    y,
                    val,
                    left=left,
                    height=bar_height * 0.9,
                    color=colors.get(tech, "#cccccc"),
                    edgecolor="white",
                    linewidth=0.3,
                )
                left += val

    # Left axis: scenario labels
    ax.set_yticks(y_ticks_scenario)
    ax.set_yticklabels(y_tick_scenario_labels, fontsize=9)

    # Right axis: model labels
    ax2 = ax.twinx()
    ax2.set_ylim(ax.get_ylim())
    ax2.set_yticks(y_ticks_model)
    ax2.set_yticklabels(models, fontsize=11, fontweight="bold")

    # X axis
    ax.set_xlabel(x_label, fontsize=11)
    ax.grid(axis="x", linestyle="--", alpha=0.4)
    ax.set_axisbelow(True)

    # Invert y so first model is on top
    ax.invert_yaxis()
    ax2.invert_yaxis()

    # Legend (outside, below)
    handles = [
        mpatches.Patch(facecolor=colors.get(t, "#cccccc"), label=t) for t in techs
    ]
    ax.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.08),
        ncol=min(6, len(techs)),
        fontsize=8,
        frameon=False,
    )

    fig.tight_layout()
    return fig